# Mediterranean Marine Heatwaves — MHEAT tutorial

## Scientific context

A **Marine Heatwave (MHW)** is a discrete, prolonged anomalously warm event in the ocean. Formally, Hobday *et al.* (2016) defined an MHW as a period of at least **five consecutive days** during which sea-surface temperature (SST) exceeds the seasonally varying 90th percentile computed over a 30-year reference climatology (typically 1991–2020). Once detected, an event is further ranked into five severity categories (I Moderate → V Super-Extreme) by the ratio of peak anomaly to the threshold anomaly. Because the definition is relative to local climatology, the same absolute temperature can be an MHW in one region and perfectly normal in another.

The **Mediterranean Sea** is one of the ocean's fastest-warming basins — surface temperatures have risen ≈ 0.4 °C per decade since the 1980s, roughly three times the global mean, and the basin has experienced repeated basin-wide MHWs in 2003, 2015, 2017, 2018, 2022 and 2023. These events coincide with mass mortality of benthic invertebrates (gorgonians, sponges, bivalves), losses in finfish aquaculture, and large-scale bleaching of *Posidonia oceanica* seagrass meadows — a keystone Mediterranean habitat. Quantifying the co-location of extreme events with vulnerable sectors is a direct policy need under the EU Marine Strategy Framework Directive and the Biodiversity Strategy 2030.

This notebook walks through the MHEAT workflow on a **bundled synthetic SST cube** so you can run every cell **without a Copernicus account** or internet access. The same code, unchanged, works on real Copernicus Marine data once credentials are provided — see the final section.

---

## Dependencies

```bash
pip install xarray numpy pandas matplotlib marineHeatWaves netCDF4
# Optional, for live CMS fetches:
pip install copernicusmarine
```

## Run on EDITO Datalab in 60 seconds

If you just want to see MHEAT work before reading the theory:

1. Launch the **JupyterLab** service in the EDITO Datalab (any Python 3.11 image — the `minimal-python` profile works).
2. In a Datalab terminal:

   ```bash
   git clone https://github.com/your-org/mheat.git
   cd mheat
   pip install -r backend/requirements.txt
   ```

3. Re-open this notebook from `mheat/tutorials/` and run all cells. Every cell works on the **bundled synthetic SST cube** — no Copernicus credentials required.
4. When you are ready to pivot to real data, see *"Switch to live Copernicus Marine"* at the bottom of the notebook.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# Make the MHEAT backend importable. Adds any of (repo/backend, /srv/app)
# depending on whether the notebook is run from a clone or inside the
# Docker image.
for candidate in (os.path.abspath('../backend'), '/srv/app'):
    if os.path.isdir(os.path.join(candidate, 'app')) and candidate not in sys.path:
        sys.path.insert(0, candidate)
print('Python', sys.version.split()[0])

## Step 1 — Load a Mediterranean SST cube

We use the synthetic cube shipped with MHEAT. It's a 3-year (2020–2022) 0.25° daily SST field over a box covering most of the western and central Mediterranean, with a realistic seasonal cycle and a strong warm anomaly injected in July–August 2022 to mimic that summer's historic heatwave.

The cube has shape ``(1096 days × 41 lat × 73 lon)``.

In [ ]:
from app.sst import build_synthetic_med_cube

ds = build_synthetic_med_cube(anomaly_year=2022)
print(ds)

## Step 2 — Compute the seasonal climatology + 90th-percentile threshold

We pick the warmest pixel for illustration, run ``marineHeatWaves.detect()`` and plot the three reference curves: the daily SST, the smoothed day-of-year mean (climatology), and the 90th-percentile threshold. Shaded red spans are the MHWs the detector flagged.

In [ ]:
from marineHeatWaves import detect as mhw_detect

da = ds['analysed_sst']

# Warmest pixel at the 2022 peak day.
peak_day = da.sel(time='2022-07-25').mean(['latitude', 'longitude']).item()
# Actually choose the hottest pixel on that date.
peak_slice = da.sel(time='2022-07-25')
iy, ix = np.unravel_index(int(np.argmax(peak_slice.values)), peak_slice.shape)
lat_val = float(da['latitude'][iy]); lon_val = float(da['longitude'][ix])
print(f'Hottest pixel at 2022-07-25: lat={lat_val:.2f}° lon={lon_val:.2f}°')

series = da.isel(latitude=iy, longitude=ix).values.astype('float64')
times = pd.to_datetime(da.time.values)
ordinals = np.array([t.toordinal() for t in times])

mhws, clim = mhw_detect(
    ordinals, series,
    climatologyPeriod=[times[0].year, times[-1].year - 1],
    pctile=90, windowHalfWidth=5,
    minDuration=5, joinAcrossGaps=True, maxGap=2,
)
print(f"Events detected on this pixel: {mhws['n_events']}")
for i in range(mhws['n_events']):
    s = pd.Timestamp.fromordinal(int(mhws['time_start'][i]))
    e = pd.Timestamp.fromordinal(int(mhws['time_end'][i]))
    print(f"  #{i+1}: {s.date()} → {e.date()}  duration={int(mhws['duration'][i])}d  i_max={mhws['intensity_max'][i]:.2f}°C")

In [ ]:
# Plot 1 — time series with threshold bands
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(times, series, label='SST', color='black', lw=0.8)
ax.plot(times, clim['seas'], label='Climatology (DOY mean)', color='tab:blue')
ax.plot(times, clim['thresh'], label='90th percentile', color='tab:orange', ls='--')
for i in range(mhws['n_events']):
    s = pd.Timestamp.fromordinal(int(mhws['time_start'][i]))
    e = pd.Timestamp.fromordinal(int(mhws['time_end'][i]))
    ax.axvspan(s, e, color='red', alpha=0.25)
ax.set_ylabel('SST (°C)')
ax.set_title(f'Hobday MHW detection — pixel ({lat_val:.2f}°N, {lon_val:.2f}°E)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Step 3 — Pixel-wise detection + clustering with the MHEAT helpers

The backend wraps ``marineHeatWaves.detect`` in ``detect_cube`` (pixel-wise) and ``cluster_events`` (merges space-time neighbours). Together they turn thousands of tiny per-pixel events into a handful of contiguous region polygons — the right unit of analysis for impact studies.

In [ ]:
from app.mhw import detect_cube, cluster_events

events = detect_cube(da, clim_period=(2020, 2021), max_pixels=400)
print(f'Per-pixel events detected: {len(events)}')

clusters = cluster_events(events)
print(f'After clustering: {len(clusters)}')
for c in clusters[:5]:
    print(f"  {c.event_id}  {c.category_name}  {c.date_start} → {c.date_end}  n_pixels={c.n_pixels}")

## Step 4 — Spatial map of event density + intensity histogram

In [ ]:
# Plot 2 — spatial map of per-pixel event counts, derived from the detected events
grid = np.zeros((da.sizes['latitude'], da.sizes['longitude']))
lat_vals = da['latitude'].values
lon_vals = da['longitude'].values
for e in events:
    iy = int(np.argmin(np.abs(lat_vals - e.centroid_lat)))
    ix = int(np.argmin(np.abs(lon_vals - e.centroid_lon)))
    grid[iy, ix] += 1

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.pcolormesh(lon_vals, lat_vals, grid, cmap='YlOrRd', shading='auto')
plt.colorbar(im, ax=ax, label='# events at pixel')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Per-pixel MHW event count — 2020–2022 synthetic Mediterranean')
# Overlay cluster centroids as black stars
for c in clusters:
    ax.plot(c.centroid_lon, c.centroid_lat, marker='*', color='black', ms=12, mew=0.5)
plt.tight_layout(); plt.show()

In [ ]:
# Plot 3 — intensity histogram by Hobday category
fig, ax = plt.subplots(figsize=(8, 4))
intens = [e.intensity_max for e in events]
cats = [e.category for e in events]
cat_colors = {1:'#ffd166', 2:'#ff9f1c', 3:'#e63946', 4:'#9d0208', 5:'#370617'}
cat_labels = {1:'I Moderate', 2:'II Strong', 3:'III Severe', 4:'IV Extreme', 5:'V Super-Extreme'}
bins = np.linspace(0, max(intens) + 0.5, 20)
for c in sorted(set(cats)):
    sub = [i for i, cc in zip(intens, cats) if cc == c]
    ax.hist(sub, bins=bins, alpha=0.7, label=cat_labels[c], color=cat_colors[c])
ax.set_xlabel('Peak intensity (°C above threshold)')
ax.set_ylabel('# events')
ax.set_title('Distribution of per-pixel MHW peak intensities')
ax.legend()
plt.tight_layout(); plt.show()

## Step 5 — Join with sectoral overlays

The backend ships a small pack of Mediterranean aquaculture sites, Natura 2000 marine MPAs, and seagrass polygons under ``backend/app/fixtures/overlays/``. ``attach_impact_properties`` reports, per event, how many aquaculture sites are affected and how many km² of MPA / seagrass habitat the event polygon covers.

In [ ]:
import json
from pathlib import Path
from app.impact import attach_impact_properties
from app.mhw import events_to_geojson

# Find the overlay fixtures dir — either in the cloned repo or inside the container.
candidates = [Path('../backend/app/fixtures/overlays'), Path('/srv/app/app/fixtures/overlays')]
fixtures_dir = next((p for p in candidates if p.is_dir()), candidates[0])

overlays = {k: json.loads((fixtures_dir / f'{k}.json').read_text()) for k in ('aquaculture','mpa','seagrass')}

geojson = events_to_geojson(clusters)
attach_impact_properties(geojson, clusters, overlays)
for f in geojson['features'][:3]:
    p = f['properties']
    imp = p.get('impact', {})
    print(f"{f['id']}  {p['category_name']}  aquaculture={imp.get('n_aquaculture_sites')}  mpa_km2={imp.get('mpa_area_km2')}  seagrass_km2={imp.get('seagrass_area_km2')}")

## Two additional entry points — HTTP API and CLI

Everything the notebook does is also reachable via the MHEAT HTTP service (`/api/*`) and the `python -m mheat` CLI. These are the paths most Datalab users prefer once they are beyond the "understand the method" phase.

In [ ]:
# HTTP API: point to a running MHEAT service (local dev, or a deployed EDITO instance).
import os, requests
BASE = os.environ.get('MHEAT_BASE_URL', 'http://localhost:8000')
try:
    r = requests.get(f'{BASE}/api/events',
                     params={'start': '2022-07-01', 'end': '2022-08-31', 'min_category': 3},
                     timeout=10)
    r.raise_for_status()
    fc = r.json()
    print(f'{len(fc["features"])} Category-III+ events, basin-wide Jul-Aug 2022')
except Exception as e:
    print(f'MHEAT HTTP service not reachable at {BASE} — skipping ({e}).')

In [ ]:
# CLI: drives the same API in-process, handy from a Datalab terminal.
# In a terminal:
#   python -m mheat events --start 2022-07-01 --end 2022-08-31 --min-category 3 --compact
#
# Here we exec it programmatically to show the parity:
import subprocess, sys, json
proc = subprocess.run(
    [sys.executable, '-m', 'mheat', 'events',
     '--start', '2022-07-01', '--end', '2022-08-31',
     '--min-category', '3', '--compact'],
    capture_output=True, text=True, env={**os.environ, 'DEMO_MODE': 'true'},
)
if proc.returncode == 0:
    fc = json.loads(proc.stdout)
    print(f'CLI returned {len(fc["features"])} events (same catalogue as the HTTP call above).')
else:
    print(f'CLI not installed from this kernel ({proc.returncode}). Run from a terminal in the MHEAT repo.')

## Discover MHEAT on the EDITO STAC catalogue

Once MHEAT is registered, Datalab users can discover the MHW event cube via STAC the same way they discover any other EDITO dataset:

In [ ]:
import os, requests
STAC = os.environ.get('EDITO_STAC_URL', 'https://catalog.dive.edito.eu/stac')
try:
    r = requests.get(f'{STAC}/collections/mhw-events', timeout=10)
    if r.status_code == 200:
        col = r.json()
        print(col['title'])
        print('spatial:', col['extent']['spatial']['bbox'][0])
        print('temporal:', col['extent']['temporal']['interval'][0])
    else:
        print(f'Not registered yet ({r.status_code}).')
except Exception as e:
    print(f'STAC catalogue not reachable: {e}')

## How to run this on EDITO Datalab

1. In the EDITO Datalab, launch the **JupyterLab** service (any Python 3.11 image).
2. Open a terminal and `git clone https://github.com/<your-org>/mheat.git`, then `cd mheat/tutorials` and open this notebook.
3. `pip install -r ../backend/requirements.txt` from the notebook environment (or bake the deps into your image).
4. To run against **real** Copernicus Marine data, set `COPERNICUSMARINE_SERVICE_USERNAME` and `COPERNICUSMARINE_SERVICE_PASSWORD` in the notebook environment and replace the synthetic load with:

```python
import copernicusmarine
out = copernicusmarine.subset(
    dataset_id='SST_MED_SST_L4_NRT_OBSERVATIONS_010_004',
    minimum_longitude=-6, maximum_longitude=36.5,
    minimum_latitude=30, maximum_latitude=46,
    start_datetime='2022-05-01T00:00:00', end_datetime='2022-09-30T23:59:59',
    variables=['analysed_sst'],
)
ds = xr.open_dataset(out)
```

5. The rest of the notebook runs unchanged. The MHEAT service itself (``docker compose up --build``) is also re-deployable onto an EDITO **Onyxia** instance — point `FRONTEND_DIR` at the built Vite bundle and expose port 8000.

## Citations

* Hobday, A. J., Alexander, L. V., Perkins, S. E., Smale, D. A., *et al.* (2016). *A hierarchical approach to defining marine heatwaves.* **Progress in Oceanography**, 141, 227–238.
* Hobday, A. J., Oliver, E. C. J., *et al.* (2018). *Categorizing and naming marine heatwaves.* **Oceanography**, 31(2), 162–173.
* Oliver, E. C. J. (2019). `marineHeatWaves` — a Python package for identification of marine heat waves. https://github.com/ecjoliver/marineHeatWaves
* Copernicus Marine Service — https://marine.copernicus.eu